In [10]:
import torch
from transformers import  CamembertForMaskedLM, CamembertTokenizer, CamembertModel,RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline
from transformers import BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM 
from transformers import TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss

In [11]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)

In [33]:
#Data
source = "test/Polish.txt"
file = open(source, "r", encoding = 'utf-8')
lines = file.readlines()

data = []

for l in range(14):
    line = lines[l]
    
    data.append(line)

In [34]:
def bert_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = "[CLS] " + text + " [SEP]" #special token for BERT, RoBERTa
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    length = len(tokenized_text)-2
    for masked_index in range(1,len(tokenized_text)-1):
        # Mask a token that we will try to predict back with `BertForMaskedLM`
        masked_word = tokenized_text[masked_index]
        #tokenized_text[masked_index] = '<mask>' #special token for XLNet
        tokenized_text[masked_index] = '[MASK]' #special token for BERT, RoBerta
        # Convert token to vocabulary indices
        indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
        index = torch.tensor(tokenizer.convert_tokens_to_ids(masked_word))
        tokens_tensor = torch.tensor([indexed_tokens])
        tokens_tensor = tokens_tensor.to('cuda')
        index = index.to('cuda')
        #masked_tensor = torch.tensor([masked_index])
        with torch.no_grad():
            outputs = model(tokens_tensor.to('cuda'))
        prediction_scores = outputs[0]
        prediction_scores = prediction_scores.view(-1, model.config.vocab_size)
        prediction_scores = prediction_scores[masked_index].unsqueeze(0)
        loss_fct = CrossEntropyLoss(ignore_index=-1)  # -1 index = padding token
        masked_lm_loss = loss_fct(prediction_scores, index.view(-1))
        tokenized_text[masked_index] = masked_word
        sentence_score -= masked_lm_loss.item()
        tokenized_text[masked_index] = masked_word
    sentence_score = sentence_score/length
    return sentence_score

In [35]:
def uni_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = text
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    length = len(tokenized_text)
    tokens_tensor = torch.tensor([indexed_tokens])
    tokens_tensor = tokens_tensor.to('cuda')
    #masked_tensor = torch.tensor([masked_index])
    with torch.no_grad():
        outputs = model(tokens_tensor, labels= tokens_tensor)
    loss = outputs[0]
    sentence_score = -loss
    return sentence_score

In [38]:
def score_model(model, tokenizer, data):
    opts = ["stały","dzieci","dalszy","trudno","bawi","krajach","pieniądze","kosztowne","młode","wierzą","osiąga","trenowania", "stały","dzieci"]
    #opts = ["vyrástli","deti","druhoradé","ťažké","hrá","krajinách","peniaze","drahé","nevyvinuté","veria","dostane","trénovania"]
    for d in data:
        print(d)
        print("Correct option is: ", opts[data.index(d)])
        scores = {}
        for o in opts:
            sentence = d.replace("{}", o)
            scores.update({o : uni_predict(sentence, model, tokenizer)})
        scores = sorted(scores.items(), key=lambda x: x[1], reverse = True)
        for key, value in scores:
            print(key, ':', value)
        print()
        

In [39]:
#Polish
#print(torch.cuda.is_available())
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
#print(bert_predict("text text text", model, tokenizer))
scoredModel = score_model(model, tokenizer, data)

Some weights of the model checkpoint at dkleczek/bert-base-polish-uncased-v1 were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
stały : tensor(-0.1064, device='cuda:0')
osiąga : tensor(-0.1135, device='cuda:0')
trenowania : tensor(-0.2724, device='cuda:0')
bawi : tensor(-0.2894, device='cuda:0')
dzieci : tensor(-0.3962, device='cuda:0')
wierzą : tensor(-0.4806, device='cuda:0')
pieniądze : tensor(-0.5181, device='cuda:0')
młode : tensor(-0.5759, device='cuda:0')
krajach : tensor(-0.6267, device='cuda:0')
kosztowne : tensor(-0.6311, device='cuda:0')
trudno : tensor(-0.6466, device='cuda:0')
dalszy : tensor(-0.8302, device='cuda:0')

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
młode : tensor(-0.2209, device='cuda:0')
dzieci : tensor(-0.2396, device='cuda:0')
trenowania : tensor(-0.2648, device='cuda:0')
pieniądze : tensor(-0.2697, devic

stały : tensor(-0.4115, device='cuda:0')
osiąga : tensor(-0.4198, device='cuda:0')
trenowania : tensor(-0.4405, device='cuda:0')
bawi : tensor(-0.4421, device='cuda:0')
dzieci : tensor(-0.4570, device='cuda:0')
pieniądze : tensor(-0.4702, device='cuda:0')
wierzą : tensor(-0.4750, device='cuda:0')
młode : tensor(-0.4774, device='cuda:0')
krajach : tensor(-0.4892, device='cuda:0')
kosztowne : tensor(-0.4918, device='cuda:0')
trudno : tensor(-0.4937, device='cuda:0')
dalszy : tensor(-0.5184, device='cuda:0')

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by _ się one najlepszymi sportowcami?  Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku. Praca w szkole, spotykanie się ze znajomymi I inne zainteresowania muszą zejść na _ plan. Bardzo _ jest wyjaśnić małym dzieciom dlaczego muszą trenować pięć godzin dziennie. To dotyczy także weekendów, kiedy większość 

In [40]:
#Czech 2
#Uses BERT tokenizer to avoid sentencepiece "not a string" error
tokenizer = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
model = AlbertForMaskedLM.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True).cuda()
scoredModel = score_model(model, tokenizer, data)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
All TF 2.0 model weights were used when initializing AlbertForMaskedLM.

Some weights of AlbertForMaskedLM were not initialized from the TF 2.0 model and are newly initialized: ['predictions.decoder.weight', 'predictions.decoder.bias', 'predictions.decoder.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
trudno : tensor(-0.6725, device='cuda:0')
bawi : tensor(-0.6728, device='cuda:0')
kosztowne : tensor(-0.6744, device='cuda:0')
krajach : tensor(-0.6802, device='cuda:0')
trenowania : tensor(-0.6828, device='cuda:0')
pieniądze : tensor(-0.6922, device='cuda:0')
dalszy : tensor(-0.6976, device='cuda:0')
młode : tensor(-0.6997, device='cuda:0')
osiąga : tensor(-0.7263, device='cuda:0')
stały : tensor(-0.7270, device='cuda:0')
dzieci : tensor(-0.7315, device='cuda:0')
wierzą : tensor(-0.7505, device='cuda:0')

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
pieniądze : tensor(-1.0340, device='cuda:0')
stały : tensor(-1.0980, device='cuda:0')
wierzą : tensor(-1.0980, device='cuda:0')
dalszy : tensor(-1.1636, device='c

RuntimeError: The expanded size of the tensor (520) must match the existing size (512) at non-singleton dimension 1.  Target sizes: [1, 520].  Tensor sizes: [1, 512]

In [41]:
tokenizer = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
model = RobertaForMaskedLM.from_pretrained('gerulata/slovakbert').cuda()
scoredModel = score_model(model, tokenizer, data)

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
bawi : tensor(-0.5924, device='cuda:0')
stały : tensor(-0.6042, device='cuda:0')
kosztowne : tensor(-0.6042, device='cuda:0')
pieniądze : tensor(-0.6159, device='cuda:0')
dalszy : tensor(-0.6219, device='cuda:0')
trudno : tensor(-0.6245, device='cuda:0')
dzieci : tensor(-0.6308, device='cuda:0')
trenowania : tensor(-0.6530, device='cuda:0')
wierzą : tensor(-0.6578, device='cuda:0')
młode : tensor(-0.6780, device='cuda:0')
osiąga : tensor(-0.7316, device='cuda:0')
krajach : tensor(-0.7799, device='cuda:0')

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
pieniądze : tensor(-1.0840, device='cuda:0')
dalszy : tensor(-1.1505, device='cuda:0')
osiąga : tensor(-1.1516, device='cuda:0')
wierzą : tensor(-1.1555, device='

RuntimeError: The expanded size of the tensor (564) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 564].  Tensor sizes: [1, 514]

In [42]:
tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-1.4B")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-1.4B").cuda()
scoredModel = score_model(model, tokenizer, data)

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
stały : tensor(-2.7270, device='cuda:0')
wierzą : tensor(-2.8120, device='cuda:0')
pieniądze : tensor(-2.8189, device='cuda:0')
dzieci : tensor(-2.8311, device='cuda:0')
trudno : tensor(-2.8629, device='cuda:0')
kosztowne : tensor(-2.8890, device='cuda:0')
młode : tensor(-2.8895, device='cuda:0')
trenowania : tensor(-2.8931, device='cuda:0')
dalszy : tensor(-2.9277, device='cuda:0')
osiąga : tensor(-2.9297, device='cuda:0')
bawi : tensor(-2.9317, device='cuda:0')
krajach : tensor(-3.0210, device='cuda:0')

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
młode : tensor(-3.2715, device='cuda:0')
dzieci : tensor(-3.3320, device='cuda:0')
pieniądze : tensor(-3.3950, device='cuda:0')
osiąga : tensor(-3.4117, device='c

stały : tensor(-2.6697, device='cuda:0')
pieniądze : tensor(-2.6835, device='cuda:0')
wierzą : tensor(-2.6862, device='cuda:0')
dzieci : tensor(-2.6887, device='cuda:0')
młode : tensor(-2.6906, device='cuda:0')
trudno : tensor(-2.6933, device='cuda:0')
trenowania : tensor(-2.6989, device='cuda:0')
kosztowne : tensor(-2.6991, device='cuda:0')
dalszy : tensor(-2.7005, device='cuda:0')
bawi : tensor(-2.7027, device='cuda:0')
osiąga : tensor(-2.7051, device='cuda:0')
krajach : tensor(-2.7171, device='cuda:0')

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by _ się one najlepszymi sportowcami?  Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku. Praca w szkole, spotykanie się ze znajomymi I inne zainteresowania muszą zejść na _ plan. Bardzo _ jest wyjaśnić małym dzieciom dlaczego muszą trenować pięć godzin dziennie. To dotyczy także weekendów, kiedy większość 